In [39]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "suda2005piagetian")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "BCO_Disc_Liq_Data.sav")
complete_path_2 = os.path.join(original_data_pathway, "discrete_conservation_follow_up_raw_data.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [40]:
import pandas as pd
import numpy as np
import pyreadstat


df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'BCO_Disc_Liq_Data.csv')
# df1.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)

df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'discrete_conservation_follow_up_raw_data.csv')
# df2.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)

df1['experiment'] = "1"
df2['experiment'] = "2"
df1.rename(columns={"NAME": "subject"}, inplace=True)

df1['AGE'].replace(1, 'non-adult', inplace=True, regex=True)
df1['AGE'].replace(2, 'adult', inplace=True, regex=True)
# df1['AGE'].unique()


In [41]:
df2['DATE'] = df2['DATE'].astype(str)

df2['year'] = df2['DATE'].str.slice(0,2)
df2['year'] = '20' + df2['year'].astype(str)
df2['month'] = df2['DATE'].str.slice(2,4)
df2['day'] = df2['DATE'].str.slice(4,6)
# df2['DATE'].unique()

# df2['year'].replace('200n', np.nan, inplace=True)
# df2['month'].replace('an', np.nan, inplace=True)

In [42]:
df1.columns
df1_temp1 = df1[['subject', 'NO_NAME', 'AGE',  'CHOICE1','CS','CDLC', 'CDLT','CD','OS','ODLC', 'ODLT','OD', 
       'experiment']].copy()
df1_temp1['publication_year'] = '2004'
df1_temp1['reward'] = 'juice'

df1_temp2 = df1[['subject', 'NO_NAME', 'AGE','DISC_1','D_CS','D_CDLC', 'D_CDLT','D_CD','D_OS','D_ODLC', 'D_ODLT','D_OD', 
       'experiment']].copy()
df1_temp2['publication_year'] = '2005'
df1_temp2['reward'] = 'cereal_or_raisins'

df1_temp = df1_temp1.values.tolist() + df1_temp2.values.tolist()

df3_1 = pd.DataFrame(df1_temp, columns=['subject','trial','age', 'overall_success', 'CS_condition_success','LC_of_CD_condition_success','LT_of_CD_condition_success', 'CD_condition_success',
                                        'OS_condition_success','LC_of_OD_condition_success', 'LT_of_OD_condition_success','OD_condition_success',
       'experiment', 'publication_year','reward'])

In [43]:
# df3_1['subject'].unique()
df3_1.loc[df3_1.subject == 'Ulla', ['drop_out']] = 'true'
df3_1.loc[df3_1.subject == 'Sandra', ['drop_out']] = 'true'

In [44]:
data_frames=[df3_1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "participant",
                      "species": "species_original",
                      "age":"age_original"}, inplace=True) ##standardize names for participants
    x['participant'] = x['participant'].str.rstrip() ##remove spaces
    x['study_id']="suda2005piagetian"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [45]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [46]:
fulldf.rename(columns={ "type": "trial_type",
                       "lclt":"lc_or_lt_trial",
                       "am1":"1st_amount_chosen",
                       "am2":"2nd_amount_chosen",
                       "con2":"2nd_choice_container",
                       "fin2":"2nd_choice_hesitation"}, inplace=True) 

In [47]:
# fulldf.columns
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'].replace('nan-nan-nan', np.nan, inplace=True)

fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

fulldf['trial_type'] = fulldf['trial_type'].str.replace(': ', '_')

replace_list = ['lc_or_lt_trial', '1st_amount_chosen',
       '2nd_amount_chosen', '2nd_choice_container']
for x in replace_list:
        fulldf[x] = fulldf[x].str.replace(' ', '_')

In [48]:

fulldf=fulldf[['study_id','year','month','day','publication_year','experiment','participant', 'age_original','age_in_years', 
                             'sex','species', 'trial',
                              #  'number',
                               'trial_type','reward',  'overall_success',
       'cs_condition_success', 'lc_of_cd_condition_success',
       'lt_of_cd_condition_success', 'cd_condition_success',
       'os_condition_success', 'lc_of_od_condition_success',
       'lt_of_od_condition_success', 'od_condition_success', 
         'lc_or_lt_trial', '1st_amount_chosen',
       '2nd_amount_chosen', '2nd_choice_container', '2nd_choice_hesitation', 'drop_out']]

for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'suda2005piagetian_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'suda2005piagetian_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)